# Version 2 — Réseau profond, régularisation, interprétabilité
**Responsable : FALL Madjiguène** · relecture : DIOP Alioune

Parties 3, 4, 5 et 7 du sujet (la Partie 6 réutilise `interpret.py`, écrit par Alioune).

**Organisation.** Les expériences longues (P3, P4, P7) ont été lancées avec les scripts de `scripts/`, qui enregistrent
tous les entraînements (3 graines, historiques complets) dans `results/logs/`. Ce notebook **relit ces résultats**
pour construire les tableaux (`RECALCULER = False`), puis réentraîne les configurations retenues (contrôle de
cohérence en 5.5), trace les courbes et les projections, et évalue **une seule fois** sur le test (section 7).
Avec `RECALCULER = True`, les scripts sont relancés (environ 1 h 30 sur CPU).

## 1. Imports

In [ ]:
import sys; sys.path.append("../src")
import json, subprocess
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import torch
from torch import nn
from har import config, data, models, training, metrics, plots, interpret
training.set_seed(0)
print("device :", training.DEVICE)

RECALCULER = False          # True : relance les scripts P3, P4, P7 (long)

def cached(name, script):
    """Relit results/logs/<name>.json ; si absent (ou RECALCULER), relance le script qui le produit."""
    path = config.LOG_DIR / f"{name}.json"
    if RECALCULER or not path.exists():
        subprocess.run([sys.executable, f"scripts/{script}"], cwd=config.ROOT, check=True)
    return json.loads(path.read_text())

def summarize(logs, keys=None):
    """Tableau moyenne ± écart-type (3 graines) à partir des logs de run_seeds."""
    rows = []
    for k in (keys or logs):
        s = logs[k]["summary"]
        hs = [r["history"] for r in logs[k]["runs"]]
        rows.append({"config": k,
                     "F1 val": f"{s['val_f1']['mean']:.4f} ± {s['val_f1']['std']:.4f}",
                     "acc val": f"{s['val_acc']['mean']:.4f} ± {s['val_acc']['std']:.4f}",
                     "Δ train−val": round(np.mean([h["train_acc"][-1] for h in hs]) - s["val_acc"]["mean"], 3),
                     "temps (s)": round(np.mean([h["train_time_s"] for h in hs]), 1),
                     "f1_mean": s["val_f1"]["mean"], "f1_std": s["val_f1"]["std"]})
    return pd.DataFrame(rows)

## 2. Chargement et exploration des données

In [ ]:
df = data.load_train()
df.shape

## 3. Découpage apprentissage / validation et DataLoaders

In [ ]:
df_tr, df_va, val_subjects = data.split_by_subject(df)
print("volontaires de validation :", val_subjects)
assert set(df_tr.subject) & set(df_va.subject) == set()
X_tr, y_tr, X_va, y_va, scaler = data.prepare(df_tr, df_va)
train_loader, val_loader = data.make_loaders(X_tr, y_tr, X_va, y_va, seed=0)
IN_DIM = X_tr.shape[1]
X_tr.shape, X_va.shape

## 4. Modèles et fonctions d'entraînement / évaluation
`models.MLP` (largeur, profondeur, activations par couche, dropout, BatchNorm) et `training.run_seeds`
(3 graines, arrêt anticipé optionnel) sont dans `src/har`. Nombre de paramètres des architectures étudiées :

In [ ]:
ARCHS = {"[64]": [64], "[128]": [128], "[256]": [256],
         "2 couches": [128, 64], "3 couches": [256, 128, 64], "4 couches": [256, 128, 64, 32]}
pd.Series({k: models.count_params(models.MLP(IN_DIM, h)) for k, h in ARCHS.items()}, name="paramètres").to_frame()

## 5. Boucle d'entraînement principale
### 5.1 Largeur de la couche cachée (P3.1)
Script `scripts/p31_width.py` : largeurs 64, 128, 256 ; pas d'apprentissage dans {1e-4, 3e-4, 1e-3, 3e-3} ;
Adam, 30 époques, 3 graines. Pour chaque largeur, on garde le meilleur pas (F1 macro de validation).

In [ ]:
w = cached("p31_width_lr_grid", "p31_width.py")
t31 = summarize(w)
t31["largeur"] = t31["config"].str.extract(r"w(\d+)_")[0].astype(int)
t31["lr"] = t31["config"].str.extract(r"lr([\d.e-]+)")[0].astype(float)
t31["paramètres"] = [models.count_params(models.MLP(IN_DIM, [x])) for x in t31["largeur"]]
best31 = t31.loc[t31.groupby("largeur")["f1_mean"].idxmax()]
display(best31[["largeur", "paramètres", "lr", "F1 val", "acc val", "Δ train−val", "temps (s)"]])
print(f"écart maximal de F1 entre largeurs : {best31['f1_mean'].max() - best31['f1_mean'].min():.4f} "
      f"(écarts-types : {best31['f1_std'].min():.4f} à {best31['f1_std'].max():.4f})")

**Lecture.** Passer de 64 à 256 neurones multiplie les paramètres par quatre sans changer le F1 de validation de
façon détectable (écart inférieur aux écarts-types entre graines), alors que l'écart apprentissage − validation
augmente : un réseau plus large sur-apprend davantage sans mieux généraliser. Le meilleur pas est le plus petit de la
grille pour 64 et 128 : avec 30 époques, un petit pas joue en partie le rôle d'un arrêt anticipé.

### 5.2 Profondeur et combinaisons d'activations (P3.2)
Script `scripts/p32_depth_activation.py` : 2, 3 et 4 couches ; activations ReLU, tanh, sigmoïde, et ReLU avec tanh
sur la dernière couche cachée ; pas dans {1e-4, 3e-4, 1e-3} ; 30 époques, 3 graines.

In [ ]:
d = cached("p32_depth_activation", "p32_depth_activation.py")
t32 = summarize(d)
parts = t32["config"].str.extract(r"^(?P<profondeur>.+?)_(?P<activation>.+)_lr(?P<lr>[\d.e-]+)$")
t32 = pd.concat([parts, t32], axis=1)
t32["paramètres"] = [models.count_params(models.MLP(IN_DIM, ARCHS[p])) for p in t32["profondeur"]]
best32 = t32.loc[t32.groupby(["profondeur", "activation"])["f1_mean"].idxmax()].sort_values("f1_mean", ascending=False)
display(best32[["profondeur", "activation", "lr", "paramètres", "F1 val", "Δ train−val", "temps (s)"]])
for p in ["2 couches", "3 couches", "4 couches"]:
    sub = best32[(best32.profondeur == p) & (best32.activation != "sigmoid")]
    print(f"{p} : écart de F1 entre activations (hors sigmoïde) = {sub.f1_mean.max() - sub.f1_mean.min():.4f}, "
          f"écarts-types {sub.f1_std.min():.4f} à {sub.f1_std.max():.4f}")

**Lecture.** Ajouter des couches n'améliore pas la validation (meilleur F1 : 2 couches > 3 > 4), avec un écart
d'apprentissage et un temps croissants. À profondeur donnée, les activations ne se départagent pas (écarts du même
ordre que les écarts-types). Seul effet net : 4 couches avec sigmoïde n'apprend pas (F1 ≈ 0,18), vraisemblablement à
cause de l'atténuation des gradients à travers des sigmoïdes empilées (explication non vérifiée).
Pour la suite, on retient [256, 128, 64] avec ReLU, ReLU, tanh (pas 3·10⁻⁴), défendable mais non démontré meilleur.

### 5.3 Régularisation (P4)
Script `scripts/p4_regularisation.py` : sur [256, 128, 64] (ReLU, ReLU, tanh ; pas 3·10⁻⁴ ; 30 époques ; 3 graines),
chaque technique isolément puis combinée. Pour les lignes avec arrêt anticipé, scores et loss sont pris à la meilleure
époque (choisie sur la validation, donc légèrement optimistes).

In [ ]:
reg = cached("p4_regularisation", "p4_regularisation.py")
t4 = pd.read_csv(config.RESULTS_DIR / "p4_regularisation.csv")
display(t4)
base = reg["base (aucune)"]["summary"]["val_f1"]
for k in reg:
    s = reg[k]["summary"]["val_f1"]
    verdict = "significatif" if abs(s["mean"] - base["mean"]) > s["std"] + base["std"] else "non significatif"
    print(f"{k:26s} ΔF1 vs base = {s['mean'] - base['mean']:+.4f} (s1+s2 = {s['std'] + base['std']:.4f}) -> {verdict}")

**Lecture.** Aucune technique seule (hors arrêt anticipé) n'améliore le F1 de façon détectable ; dropout 0,5,
BatchNorm et arrêt anticipé réduisent en revanche la loss de validation (le réseau est moins sûr de lui à accuracy
quasi constante). La combinaison BatchNorm + dropout 0,3 + weight decay + arrêt anticipé est la meilleure ; ses
variantes sans arrêt anticipé restent au niveau de la base, ce qui suggère que l'arrêt anticipé en est l'ingrédient
principal (hypothèse non isolée par nos essais).

### 5.4 Bonus : recherche automatique d'hyperparamètres (P7)
Script `scripts/p7_random_search.py` : 30 configurations tirées au hasard (architecture, activation, pas, dropout,
weight decay, BatchNorm, taille de lot), une graine (100) et arrêt anticipé ; puis ré-évaluation des 3 meilleures sur
les graines habituelles, pour limiter le biais de sélection.

In [ ]:
if RECALCULER or not (config.RESULTS_DIR / "p7_top3_reeval.csv").exists():
    subprocess.run([sys.executable, "scripts/p7_random_search.py"], cwd=config.ROOT, check=True)
p7 = pd.read_csv(config.RESULTS_DIR / "p7_random_search.csv")
display(p7.head(5))
display(pd.read_csv(config.RESULTS_DIR / "p7_top3_reeval.csv"))

**Lecture.** Le meilleur score à une graine retombe une fois ré-évalué sur 3 graines (effet attendu de la sélection
parmi 30 essais) ; les trois meilleures configurations restent au niveau de la combinaison retenue en 5.3 : la
recherche aléatoire ne trouve rien de significativement meilleur.

### 5.5 Configurations retenues et contrôle de cohérence
Trois configurations sont retenues sur la validation (`results/selected_configs.json`) : `combo` (BatchNorm +
dropout 0,3 + weight decay + arrêt anticipé), `es` (arrêt anticipé seul) et `dropout5` (dropout 0,5).
On les **réentraîne** ici (mêmes graines) : les F1 de validation doivent redonner ceux de P4, ce qui vérifie que les
modèles évalués au test sont bien ceux qui ont été sélectionnés.

In [ ]:
selected = json.loads((config.RESULTS_DIR / "selected_configs.json").read_text())

def build_selected(cfg):
    def fn(seed):
        tl, vl = data.make_loaders(X_tr, y_tr, X_va, y_va, seed=seed)
        m = models.MLP(IN_DIM, cfg["hidden"], activation=cfg["activation"], dropout=cfg["dropout"], norm=cfg["norm"])
        return m, tl, vl, training.make_optimizer(cfg["optimizer"], m.parameters(), lr=cfg["lr"],
                                                  weight_decay=cfg["weight_decay"])
    return fn

final_runs = {}
for name, cfg in selected.items():
    final_runs[name] = training.run_seeds(build_selected(cfg), seeds=cfg["seeds"], keep_models=True,
                                          epochs=cfg["epochs"], patience=cfg["patience"])
    new, old = final_runs[name]["summary"]["val_f1"], cfg["val_summary"]["val_f1"]
    print(f"{name:9s} F1 val réentraîné {new['mean']:.4f} ± {new['std']:.4f} | P4 : {old['mean']:.4f} ± {old['std']:.4f} "
          f"| écart {abs(new['mean'] - old['mean']):.4f}")

## 6. Visualisation et comparaison des résultats
### 6.1 Courbes d'apprentissage (P5.1)

In [ ]:
for name in selected:
    h = final_runs[name]["runs"][0]["history"]
    plots.learning_curves(h, title=f"{name} (graine 0, meilleure époque = {h['best_epoch']})", save=f"v2_curves_{name}")
hist0 = {name: final_runs[name]["runs"][0]["history"] for name in selected}
plots.compare_curves(hist0, key="val_f1", save="v2_compare_f1")
plots.compare_curves(hist0, key="val_loss", save="v2_compare_loss")
plt.show()

**Lecture.** Sans régularisation forte, la loss de validation remonte après quelques époques alors que le F1 reste
stable : le réseau devient trop sûr de ses prédictions. La combinaison garde la loss la plus basse. Aucune technique
ne réduit l'instabilité du F1 d'une époque à l'autre (`scripts/p51_stability.py`).

### 6.2 t-SNE / UMAP de la dernière couche cachée (P6)
Fenêtres de **validation** (le test reste réservé à la section 7), modèle `combo`, graine 0.

In [ ]:
final_model = final_runs["combo"]["runs"][0]["model"]
interpret.compare_spaces(final_model, X_va, y_va, method="tsne", perplexity=30, save="v2_tsne")
if interpret.umap_available():
    interpret.compare_spaces(final_model, X_va, y_va, method="umap", save="v2_umap")
H_va = interpret.hidden_activations(final_model, X_va)
display(interpret.separation_scores({"entrées": X_va, "couche cachée (combo)": H_va}, y_va).round(3))

**Lecture.** Dans la couche cachée, les activités dynamiques et « allongé » forment des groupes séparés ; « assis » et
« debout » se recouvrent partiellement, en accord avec la confusion principale observée au test. Les distances entre
groupes et leurs tailles ne sont pas interprétables (voir `06_interpretability.ipynb` pour la perplexité et les graines).

## 7. Évaluation finale sur le jeu de test (P5.2) — une seule fois
Configurations figées (section 5.5) ; c'est le **seul** appel à `load_test()` de la version 2.
Pour chaque configuration : F1 de validation et de test sur les 3 graines, F1 par classe, et matrice de confusion
cumulée sur les 3 graines, normalisée par ligne. (La référence et la V1 sont évaluées dans la V1.)

In [ ]:
df_test = data.load_test()
X_te_raw, y_te = data.to_xy(df_test)
X_te = scaler.transform(X_te_raw).astype(np.float32)
test_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(torch.as_tensor(X_te), torch.as_tensor(y_te)), batch_size=512)
crit = nn.CrossEntropyLoss()

rows, per_class, cms, test_results = [], {}, {}, {}
for name in selected:
    evals = [training.evaluate(r["model"], test_loader, crit, return_preds=True) for r in final_runs[name]["runs"]]
    f1_te = [e["f1_macro"] for e in evals]; acc_te = [e["accuracy"] for e in evals]
    f1_va = final_runs[name]["summary"]["val_f1"]
    rows.append({"configuration": name, "F1 val": f"{f1_va['mean']:.4f} ± {f1_va['std']:.4f}",
                 "F1 test": metrics.mean_std(f1_te), "acc test": metrics.mean_std(acc_te),
                 "Δ test − val": round(np.mean(f1_te) - f1_va["mean"], 4)})
    per_class[name] = pd.DataFrame([metrics.per_class_f1(e["y_true"], e["y_pred"]) for e in evals]).mean()
    cms[name] = sum(metrics.confusion(e["y_true"], e["y_pred"]) for e in evals)
    test_results[name] = {"f1_test": [float(x) for x in f1_te], "acc_test": [float(x) for x in acc_te],
                          "f1_val": f1_va, "f1_par_classe": per_class[name].to_dict(), "confusion_cumulee": cms[name].tolist()}
display(pd.DataFrame(rows))
display(pd.DataFrame(per_class).round(3))
for name in selected:
    plots.confusion_heatmap(cms[name], f"{name} — test (3 graines cumulées)", save=f"v2_cm_test_{name}")
training.save_json(test_results, config.RESULTS_DIR / "v2_test_results.json")

**Classes les plus difficiles.** « Assis » et « debout » ont les F1 les plus bas et se confondent mutuellement :
deux postures statiques dont les signaux d'accéléromètre et de gyroscope sont très proches, seule une faible
différence d'inclinaison du tronc les séparant (explication plausible, non vérifiée). Parmi les marches, « montée »
et « descente » se confondent entre elles et avec la marche, des mouvements périodiques voisins.

**Validation et test.** Les trois configurations perdent du F1 au test : la validation (5 volontaires) est optimiste.
Une partie peut venir de la sélection sur la validation (arrêt anticipé, choix des configurations), une autre du
changement de volontaires ; nos résultats ne permettent pas de séparer les deux effets.